not investment advice

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf

In [3]:
np.random.seed(42)

In [4]:
sp500 = list(pd.read_csv('sp500_companies.csv')['Symbol'])

In [5]:
for tick in ['ANSS', 'DFS', 'JNPR', 'WBA', 'HES', 'PARA', 'GEV', 'SOLV', 'AMTM']:
    sp500.remove(tick)

In [6]:
# ticker = sp500[np.random.randint(len(sp500))]
ticker = 'BA'
stock = yf.Ticker(ticker)

In [7]:
ticker

'BA'

In [8]:
# assert 'freeCashflow' in stock.info.keys()
# assert 'beta' in stock.info.keys()
# assert 'revenueGrowth' in stock.info.keys()
# assert 'marketCap' in stock.info.keys()

### Intrinsic Value
Intrinsic value aims to estimate what a firm is fundamentally worth based on its ability to generate future cash flows, adjusted for risk. However, valuation becomes most actionable when intrinsic value is compared directly to market prices. This comparison produces a valuation gap, a quantitative measure of how far market perception deviates from fundamental value.

In [9]:
def intrinsic_value(fcf, beta, growth, rf=0.05, equity_risk_premium=0.06, max_growth=0.05):
    g = min(growth, max_growth)
    r = rf + beta * equity_risk_premium
    
    if r <= g:
        return np.nan
    
    return fcf*(1+g)/(r-g)

In [10]:
fcf = stock.info['freeCashflow']
beta = stock.info['beta']
growth = stock.info['revenueGrowth']
market_cap = stock.info['marketCap']
intrinsic = intrinsic_value(fcf, beta, growth)
valuation_gap = np.log(market_cap / intrinsic)

C:\Users\Cassidy\AppData\Local\Temp\ipykernel_4024\3953821374.py:6: RuntimeWarning: invalid value encountered in log
  valuation_gap = np.log(market_cap / intrinsic)


In [11]:
print(f"Intrinsic Value: ${intrinsic:,.0f}")
print(f"Market Cap: ${market_cap:,.0f}")
print(f"Log Valuation Gap: {valuation_gap:.3f}")

Intrinsic Value: $-71,668,366,638
Market Cap: $191,500,075,008
Log Valuation Gap: nan


In [12]:
top10 = sp500[:10]
random10 = np.random.choice(sp500, 10, replace=False).tolist()

In [13]:
log_gaps = {}
try:
    for stock in random10:
        ticker = stock
        stock = yf.Ticker(ticker)
        fcf = stock.info['freeCashflow']
        beta = stock.info['beta']
        growth = stock.info['revenueGrowth']
        market_cap = stock.info['marketCap']
        intrinsic = intrinsic_value(fcf, beta, growth)
        valuation_gap = np.log(market_cap / intrinsic)
        print(f"{ticker}: Log Valuation Gap: {valuation_gap:.3f}")
        log_gaps[ticker] = valuation_gap
except KeyError:
    print(f"Data missing for {ticker}, skipping...")

UHS: Log Valuation Gap: 0.116
Data missing for KKR, skipping...


In [14]:
sorted(log_gaps.items(), key=lambda item: item[1])

[('UHS', np.float64(0.11561699867015186))]

### Price-to-Earnings
- How much the market is willing to pay for $1 of current earnings
- Low P/E -> market expects low growth, high risk, or temporary earnings
- High P/E -> market expects strong growth or stable earnings
- distorted by accounting choices
- meaningless for firms with negative earnings
- P/E = Price / EPS

In [17]:
stock.info['trailingPE']

54.65546

### EV / EBITDA
**What it measures**
- Firm value relative to operating cash flow proxy.

**Interpretation**
- Low EV/EBITDA → cheap on an enterprise basis
- High EV/EBITDA → market pricing in growth or low risk

**How to use it**
- Best for comparing firms with different capital structures
- Preferred in M&A and private equity

**Common pitfalls**
- Ignores capital expenditures
- EBITDA can overstate cash generation

In [25]:
stock.info['enterpriseValue']

177561780224

In [20]:
stock.info['previousClose'] / stock.info['trailingEps']

56.28991596638656

In [19]:
stock.info['priceToBook']

4.26003

In [ ]:
# I want to research asset allocation optimization, other ways to use data science in finance
# python for finance book - o'reilly

In [67]:
#### not investment advice!!!